# Experiment 2: Data Augmentation

Goal:

Determine whether data augmentation improves
generalization on FER2013.

Changes from baseline:

- Added image augmentation during training

Everything else remains identical.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [3]:
PROCESSED_PATH = "/content/drive/MyDrive/emotion_sentiment_fusion_detector/data/processed"

X_train = np.load(f"{PROCESSED_PATH}/X_train.npy")
X_val = np.load(f"{PROCESSED_PATH}/X_val.npy")
X_test = np.load(f"{PROCESSED_PATH}/X_test.npy")

y_train = np.load(f"{PROCESSED_PATH}/y_train.npy")
y_val = np.load(f"{PROCESSED_PATH}/y_val.npy")
y_test = np.load(f"{PROCESSED_PATH}/y_test.npy")

In [4]:
train_datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True
)

In [7]:
model = Sequential()

# Block 1
model.add(
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=(48,48,1)
    )
)

model.add(
    MaxPooling2D(
        pool_size=(2,2)
    )
)

# Block 2
model.add(
    Conv2D(
        64,
        (3,3),
        activation="relu"
    )
)

model.add(
    MaxPooling2D(
        pool_size=(2,2)
    )
)

# Block 3
model.add(
    Conv2D(
        128,
        (3,3),
        activation="relu"
    )
)

model.add(
    MaxPooling2D(
        pool_size=(2,2)
    )
)

# Classifier
model.add(Flatten())

model.add(
    Dense(
        128,
        activation="relu"
    )
)

model.add(
    Dropout(
        0.5
    )
)

model.add(
    Dense(
        7,
        activation="softmax"
    )
)


In [8]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [9]:
history = model.fit(
    train_datagen.flow(
        X_train,
        y_train,
        batch_size=64
    ),
    validation_data=(X_val, y_val),
    epochs=20
)

Epoch 1/20
359/359 ━━━━━━━━━━━━━━━━━━━━ 86s 234ms/step - accuracy: 0.2587 - loss: 1.7965 - val_accuracy: 0.3009 - val_loss: 1.7117
Epoch 2/20
359/359 ━━━━━━━━━━━━━━━━━━━━ 83s 231ms/step - accuracy: 0.3265 - loss: 1.6777 - val_accuracy: 0.3905 - val_loss: 1.5623
Epoch 3/20
359/359 ━━━━━━━━━━━━━━━━━━━━ 83s 230ms/step - accuracy: 0.3916 - loss: 1.5635 - val_accuracy: 0.4526 - val_loss: 1.4208
Epoch 4/20
359/359 ━━━━━━━━━━━━━━━━━━━━ 89s 247ms/step - accuracy: 0.4328 - loss: 1.4828 - val_accuracy: 0.4756 - val_loss: 1.3647
Epoch 5/20
359/359 ━━━━━━━━━━━━━━━━━━━━ 84s 234ms/step - accuracy: 0.4503 - loss: 1.4264 - val_accuracy: 0.4984 - val_loss: 1.3025
Epoch 6/20
359/359 ━━━━━━━━━━━━━━━━━━━━ 89s 247ms/step - accuracy: 0.4711 - loss: 1.3871 - val_accuracy: 0.5098 - val_loss: 1.2995
Epoch 7/20
359/359 ━━━━━━━━━━━━━━━━━━━━ 88s 246ms/step - accuracy: 0.4888 - loss: 1.3513 - val_accuracy: 0.5195 - val_loss: 1.2611
Epoch 8/20
359/359 ━━━━━━━━━━━━━━━━━━━━ 83s 232ms/step - accuracy: 0.4924 - loss: 1

In [10]:
test_loss, test_acc = model.evaluate(
    X_test,
    y_test
)

print(test_acc)

225/225 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.5678 - loss: 1.1402
0.5678461790084839


## Experiment 2 — Data Augmentation

Results:

| Model        | Train Acc | Val Acc | Test Acc |
| ------------ | --------- | ------- | -------- |
| Baseline     | 71.2%     | 56.0%   | 56.6%    |
| BatchNorm    | 75.3%     | 54.6%   | 55.2%    |
| Augmentation | 55.5%     | 57.0%   | 56.8%    |

<br>

Conclusion:

The lower training accuracy is expected because the model is seeing:
* rotated faces
* shifted faces
* zoomed faces
* flipped faces

instead of the original clean images.

Augmentation:

* Reduced overfitting.

* Slightly improved test accuracy.

* More robust model.